# Forecast v2 — log-returns + quantile loss + event features

Diferenças vs `07_forecast_baselines`:

- **Target**: log-returns, não preço bruto.
- **Loss**: pinball (quantile) loss em q={0.1, 0.5, 0.9} — intervalos de previsão, robusto a fat tails.
- **Horizontes**: apenas h=1 e h=7. h=30 sem features de eventos não é honesto.
- **Modelos**: naive | AR(7) | AR(7)+event | LightGBM com features (lags, vol, dow, event flags).
- **Universo**: corre por tier (1, 2, 3). Doppler/Souvenirs excluídos.
- **Walk-forward**: cutoff a cada 7 dias, expanding window, 90d teste.

A questão central: o sinal das event features (`days_since_update`, `in_operation`) bate AR(7) por margem significativa?

In [1]:
import sys; sys.path.insert(0, ".")
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

from _helpers import get_daily, list_modelable, daily_with_events, load_events

pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(50)

polars.config.Config

## 1. Universo por tier

In [2]:
# Tier 1 + 2 + 3 (cumulative). 1 ⊂ 2 ⊂ 3.
t1 = list_modelable(tier=1); print(f"Tier 1: {t1.height}")
t2 = list_modelable(tier=2); print(f"Tier 2: {t2.height}")
t3 = list_modelable(tier=3); print(f"Tier 3: {t3.height}")

# Items exclusively in each band (for per-tier reporting)
exclusive_t1 = t1
exclusive_t2 = t2.filter(~pl.col("name").is_in(t1["name"]))
exclusive_t3 = t3.filter(~pl.col("name").is_in(t2["name"]))
print(f"\nExclusive: t1={exclusive_t1.height}, t2-only={exclusive_t2.height}, t3-only={exclusive_t3.height}")

Tier 1: 22
Tier 2: 309
Tier 3: 1290

Exclusive: t1=22, t2-only=287, t3-only=981


## 2. Loss functions

**Pinball loss** at quantile `q`:
$$ L_q(y, \hat{y}) = \max(q(y - \hat{y}), (q-1)(y - \hat{y})) $$

At q=0.5 this is half the MAE. At q=0.1 we penalize over-predictions more (predict the 10th percentile); at q=0.9, under-predictions.

In [3]:
def pinball(y, yhat, q):
    err = y - yhat
    return np.maximum(q * err, (q - 1) * err)

def metrics(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    yt, yp = y_true[mask], y_pred[mask]
    if len(yt) == 0:
        return {k: np.nan for k in ["mae", "rmse", "pinball_50", "dir_acc"]}
    err = yp - yt
    return {
        "mae": float(np.mean(np.abs(err))),
        "rmse": float(np.sqrt(np.mean(err**2))),
        "pinball_50": float(np.mean(pinball(yt, yp, 0.5))),
        # On returns the sign is what we care about for trading
        "dir_acc": float(np.mean(np.sign(yp) == np.sign(yt))),
    }

## 3. Feature engineering

For each item, build a feature matrix per day:
- 7 lagged log-returns
- 5d and 30d rolling vol
- day of week
- `days_since_update`, `in_operation`

In [4]:
def build_features(name: str, min_history: int = 365) -> pl.DataFrame | None:
    df = daily_with_events(name)
    if df.height < min_history + 60:
        return None
    df = df.filter(pl.col("price") > 0).sort("date")
    p = df["price"].to_numpy()
    r = np.concatenate([[np.nan], np.diff(np.log(p))])
    df = df.with_columns(pl.Series("log_ret", r))

    lags = {f"lag_{k}": pl.col("log_ret").shift(k) for k in range(1, 8)}
    rolls = {
        "vol_5": pl.col("log_ret").rolling_std(5),
        "vol_30": pl.col("log_ret").rolling_std(30),
    }
    df = df.with_columns(**lags, **rolls)
    df = df.with_columns(
        pl.col("in_operation").cast(pl.Int8),
    )
    return df.drop_nulls()

## 4. Walk-forward backtest harness

In [5]:
HORIZONS = [1, 7]
LAG_COLS = [f"lag_{k}" for k in range(1, 8)]
FEAT_COLS = LAG_COLS + ["vol_5", "vol_30", "dow", "in_operation", "days_since_update"]

def make_targets(df: pl.DataFrame, h: int) -> np.ndarray:
    """Cumulative log-return over the next h days, aligned with index t."""
    r = df["log_ret"].to_numpy()
    out = np.full(len(r), np.nan)
    for i in range(len(r) - h):
        out[i] = float(np.sum(r[i+1:i+1+h]))
    return out

def naive_pred(_X, h):
    return np.zeros(len(_X))   # log-return = 0 → price unchanged

def ar7_fit_predict(train_y, test_X, test_y_len):
    """AR(7) on log-returns. Forecast = mean of last 7 lags weighted by AR coefs."""
    import statsmodels.api as sm
    try:
        m = sm.tsa.AutoReg(train_y, lags=7, old_names=False).fit()
        # Predict point-wise from each test_X row (treat last 7 lags as inputs)
        preds = np.zeros(test_X.height)
        coefs = m.params  # [const, ar1, ar2, ..., ar7]
        for i, row in enumerate(test_X.iter_rows(named=True)):
            lags = np.array([row[f"lag_{k}"] for k in range(1, 8)])
            preds[i] = coefs[0] + np.dot(coefs[1:], lags)
        return preds
    except Exception:
        return naive_pred(test_X, 1)

import lightgbm as lgb

def lgb_quantile_fit_predict(train_X, train_y, test_X, q: float):
    train_X_np = train_X[FEAT_COLS].to_numpy()
    test_X_np = test_X[FEAT_COLS].to_numpy()
    model = lgb.LGBMRegressor(
        objective="quantile", alpha=q,
        n_estimators=200, max_depth=5, learning_rate=0.05,
        min_child_samples=20, verbosity=-1,
    )
    model.fit(train_X_np, train_y)
    return model.predict(test_X_np)

## 5. Run backtest on a sample (tier 1 + tier 2 mix)

In [6]:
# Pick 20 items: top tier 1 by liquidity + 10 random tier 2 for diversity
import random
random.seed(42)
sample = (
    t1["name"].to_list()[:10]
    + random.sample(exclusive_t2["name"].to_list(), min(10, exclusive_t2.height))
)
print(f"Sample size: {len(sample)}")

def backtest_one(name: str) -> dict | None:
    feats = build_features(name)
    if feats is None or feats.height < 500:
        return None

    out = {"name": name, "n": feats.height, "results": []}
    test_size = 90
    step = 7
    cutoffs = list(range(max(365, feats.height - test_size), feats.height - max(HORIZONS), step))

    for h in HORIZONS:
        # Build target
        targets = make_targets(feats, h)
        feats_h = feats.with_columns(pl.Series("y", targets)).drop_nulls("y")

        rows = []
        for c in cutoffs:
            if c >= feats_h.height: continue
            train = feats_h.slice(0, c)
            test = feats_h.slice(c, 1)
            if not train.height or not test.height: continue

            X_train = train.drop("date")
            y_train = train["y"].to_numpy()
            X_test = test.drop("date")
            y_test = float(test["y"].item())

            # Models
            y_naive = float(naive_pred(X_test, h)[0])
            y_ar7 = float(ar7_fit_predict(y_train, X_test, 1)[0])
            try:
                y_lgb50 = float(lgb_quantile_fit_predict(X_train, y_train, X_test, 0.5)[0])
                y_lgb10 = float(lgb_quantile_fit_predict(X_train, y_train, X_test, 0.1)[0])
                y_lgb90 = float(lgb_quantile_fit_predict(X_train, y_train, X_test, 0.9)[0])
            except Exception:
                y_lgb50 = y_lgb10 = y_lgb90 = y_naive

            rows.append({
                "horizon": h, "actual": y_test,
                "naive": y_naive, "ar7": y_ar7,
                "lgb_q50": y_lgb50, "lgb_q10": y_lgb10, "lgb_q90": y_lgb90,
            })
        out["results"].extend(rows)
    return out

results = []
for nm in sample:
    print(f"  {nm[:60]}", flush=True)
    r = backtest_one(nm)
    if r:
        results.append(r)
print(f"\nbacktested {len(results)} items")

Sample size: 20
  MP9 | Starlight Protector (Field-Tested)
  StatTrak™ MP9 | Starlight Protector (Field-Tested)
  AK-47 | Inheritance (Field-Tested)
  StatTrak™ AK-47 | Inheritance (Field-Tested)
  AK-47 | Nightwish (Field-Tested)
  StatTrak™ AK-47 | Nightwish (Field-Tested)
  StatTrak™ MP9 | Starlight Protector (Minimal Wear)
  MP9 | Starlight Protector (Minimal Wear)
  StatTrak™ Desert Eagle | Printstream (Field-Tested)
  StatTrak™ FAMAS | Bad Trip (Field-Tested)
  AWP | Crakow! (Field-Tested)
  StatTrak™ AWP | Printstream (Well-Worn)
  AWP | Neo-Noir (Factory New)
  AWP | Asiimov (Field-Tested)
  SSG 08 | Dragonfire (Minimal Wear)
  StatTrak™ M4A1-S | Player Two (Field-Tested)
  Lt. Commander Ricksaw | NSWC SEAL
  StatTrak™ AK-47 | Frontside Misty (Battle-Scarred)
  P90 | Asiimov (Field-Tested)
  StatTrak™ Glock-18 | Water Elemental (Minimal Wear)

backtested 18 items


## 6. Métricas agregadas por modelo × horizon

In [7]:
rows = []
for item in results:
    for r in item["results"]:
        for model in ["naive", "ar7", "lgb_q50"]:
            rows.append({
                "name": item["name"], "horizon": r["horizon"], "model": model,
                "actual": r["actual"], "pred": r[model],
            })
df_results = pl.DataFrame(rows)

agg = []
for (m, h), grp in df_results.group_by(["model", "horizon"], maintain_order=True):
    res = metrics(grp["actual"].to_numpy(), grp["pred"].to_numpy())
    agg.append({"model": m, "horizon": h, **res})
print(pl.DataFrame(agg).sort(["horizon", "mae"]))

shape: (6, 6)
┌─────────┬─────────┬──────────┬──────────┬────────────┬──────────┐
│ model   ┆ horizon ┆ mae      ┆ rmse     ┆ pinball_50 ┆ dir_acc  │
│ ---     ┆ ---     ┆ ---      ┆ ---      ┆ ---        ┆ ---      │
│ str     ┆ i64     ┆ f64      ┆ f64      ┆ f64        ┆ f64      │
╞═════════╪═════════╪══════════╪══════════╪════════════╪══════════╡
│ naive   ┆ 1       ┆ 0.090241 ┆ 0.114476 ┆ 0.04512    ┆ 0.0      │
│ lgb_q50 ┆ 1       ┆ 0.090409 ┆ 0.115558 ┆ 0.045205   ┆ 0.555556 │
│ ar7     ┆ 1       ┆ 0.095264 ┆ 0.118649 ┆ 0.047632   ┆ 0.50463  │
│ lgb_q50 ┆ 7       ┆ 0.10119  ┆ 0.128321 ┆ 0.050595   ┆ 0.587963 │
│ naive   ┆ 7       ┆ 0.101209 ┆ 0.126087 ┆ 0.050604   ┆ 0.0      │
│ ar7     ┆ 7       ┆ 0.11856  ┆ 0.147978 ┆ 0.05928    ┆ 0.467593 │
└─────────┴─────────┴──────────┴──────────┴────────────┴──────────┘


## 7. Coverage do intervalo q10–q90

In [8]:
rows = []
for item in results:
    for r in item["results"]:
        in_band = (r["lgb_q10"] <= r["actual"] <= r["lgb_q90"])
        rows.append({
            "horizon": r["horizon"], "actual": r["actual"],
            "q10": r["lgb_q10"], "q50": r["lgb_q50"], "q90": r["lgb_q90"],
            "in_band": in_band,
        })
cov = pl.DataFrame(rows)
print("Cobertura empírica do intervalo [q10, q90]:")
print(
    cov.group_by("horizon").agg([
        pl.col("in_band").mean().alias("coverage_q10_q90"),
        pl.col("actual").std().alias("actual_vol"),
        ((pl.col("q90") - pl.col("q10")).mean()).alias("avg_band_width"),
    ]).sort("horizon")
)
print("\nNota: idealmente coverage ≈ 0.80. <0.80 → intervalos demasiado estreitos.")

Cobertura empírica do intervalo [q10, q90]:
shape: (2, 4)
┌─────────┬──────────────────┬────────────┬────────────────┐
│ horizon ┆ coverage_q10_q90 ┆ actual_vol ┆ avg_band_width │
│ ---     ┆ ---              ┆ ---        ┆ ---            │
│ i64     ┆ f64              ┆ f64        ┆ f64            │
╞═════════╪══════════════════╪════════════╪════════════════╡
│ 1       ┆ 0.523148         ┆ 0.114621   ┆ 0.168588       │
│ 7       ┆ 0.680556         ┆ 0.123888   ┆ 0.249034       │
└─────────┴──────────────────┴────────────┴────────────────┘

Nota: idealmente coverage ≈ 0.80. <0.80 → intervalos demasiado estreitos.


## 8. Per-item leaderboard h=1 (qual modelo ganha em cada item)

In [9]:
per_item = []
for item in results:
    for model in ["naive", "ar7", "lgb_q50"]:
        h1 = [r for r in item["results"] if r["horizon"] == 1]
        if not h1: continue
        actual = np.array([r["actual"] for r in h1])
        pred = np.array([r[model] for r in h1])
        mae = np.mean(np.abs(pred - actual))
        per_item.append({"name": item["name"][:50], "model": model, "mae": mae})

df_per = pl.DataFrame(per_item).pivot(index="name", on="model", values="mae")
df_per = df_per.with_columns(
    pl.when(pl.col("lgb_q50") < pl.min_horizontal("naive", "ar7"))
      .then(pl.lit("lgb"))
      .when(pl.col("ar7") < pl.col("naive"))
      .then(pl.lit("ar7"))
      .otherwise(pl.lit("naive"))
      .alias("winner")
)
print(df_per.sort("naive"))
print()
print("Winner distribution:")
print(df_per.group_by("winner").agg(pl.len().alias("n")).sort("n", descending=True))

shape: (18, 5)
┌────────────────────────────────────────────────────┬──────────┬──────────┬──────────┬────────┐
│ name                                               ┆ naive    ┆ ar7      ┆ lgb_q50  ┆ winner │
│ ---                                                ┆ ---      ┆ ---      ┆ ---      ┆ ---    │
│ str                                                ┆ f64      ┆ f64      ┆ f64      ┆ str    │
╞════════════════════════════════════════════════════╪══════════╪══════════╪══════════╪════════╡
│ StatTrak™ AK-47 | Frontside Misty (Battle-Scarred) ┆ 0.057292 ┆ 0.055008 ┆ 0.052534 ┆ lgb    │
│ AK-47 | Inheritance (Field-Tested)                 ┆ 0.067327 ┆ 0.070996 ┆ 0.064551 ┆ lgb    │
│ StatTrak™ AK-47 | Inheritance (Field-Tested)       ┆ 0.067327 ┆ 0.070996 ┆ 0.064551 ┆ lgb    │
│ Lt. Commander Ricksaw | NSWC SEAL                  ┆ 0.069045 ┆ 0.075733 ┆ 0.067786 ┆ lgb    │
│ P90 | Asiimov (Field-Tested)                       ┆ 0.073645 ┆ 0.085093 ┆ 0.079706 ┆ naive  │
│ AWP | Neo-Noi

## 9. Event uplift — LGB melhor quando há update recente?

In [10]:
# Para cada cutoff, juntar com a feature days_since_update e ver se MAE muda.
uplift_rows = []
for item in results:
    feats = build_features(item["name"])
    if feats is None: continue
    feats_dates = feats["date"].to_list()
    for r in item["results"]:
        if r["horizon"] != 1: continue
        # Need to align date — for simplicity attribute "recent_update" if avg days_since_update < 7 in last 30d
    # Approximation: just look at most recent days_since_update
    recent_dsu = feats["days_since_update"].tail(30).mean()
    h1 = [r for r in item["results"] if r["horizon"] == 1]
    if not h1: continue
    actual = np.array([r["actual"] for r in h1])
    mae_naive = np.mean(np.abs(actual))
    mae_lgb = np.mean(np.abs(np.array([r["lgb_q50"] for r in h1]) - actual))
    uplift_rows.append({
        "name": item["name"][:40],
        "recent_dsu": float(recent_dsu),
        "mae_naive": mae_naive, "mae_lgb": mae_lgb,
        "lgb_uplift_%": float((mae_naive - mae_lgb) / mae_naive * 100),
    })

uplift = pl.DataFrame(uplift_rows)
print("Items onde LGB bate naive por margem maior (h=1):")
print(uplift.sort("lgb_uplift_%", descending=True))

Items onde LGB bate naive por margem maior (h=1):
shape: (18, 5)
┌──────────────────────────────────────────┬────────────┬───────────┬──────────┬──────────────┐
│ name                                     ┆ recent_dsu ┆ mae_naive ┆ mae_lgb  ┆ lgb_uplift_% │
│ ---                                      ┆ ---        ┆ ---       ┆ ---      ┆ ---          │
│ str                                      ┆ f64        ┆ f64       ┆ f64      ┆ f64          │
╞══════════════════════════════════════════╪════════════╪═══════════╪══════════╪══════════════╡
│ StatTrak™ AK-47 | Frontside Misty (Battl ┆ 2.466667   ┆ 0.057292  ┆ 0.052534 ┆ 8.30539      │
│ AK-47 | Inheritance (Field-Tested)       ┆ 2.466667   ┆ 0.067327  ┆ 0.064551 ┆ 4.123105     │
│ StatTrak™ AK-47 | Inheritance (Field-Tes ┆ 2.466667   ┆ 0.067327  ┆ 0.064551 ┆ 4.123105     │
│ StatTrak™ MP9 | Starlight Protector (Min ┆ 2.433333   ┆ 0.10458   ┆ 0.101623 ┆ 2.827808     │
│ MP9 | Starlight Protector (Minimal Wear) ┆ 2.433333   ┆ 0.10458   ┆ 0

## Conclusões a tirar

Olha as 3 tabelas acima e responde:

1. **§6**: MAE de `lgb_q50` é menor que `naive` em h=1? Por quanto?
2. **§7**: Coverage do intervalo está próxima de 0.80? Se for muito menor → modelo sobre-confiante. Se muito maior → demasiado conservador.
3. **§8**: Que fração dos items o LGB ganha? Se for >60%, o features-stack tem valor real.
4. **§9**: O uplift é maior em items com `recent_dsu` baixo (updates recentes)? Se sim → as event features estão a funcionar.

**Próximo passo se h=1 ganhar claramente**: adicionar peer features (média de returns do mesmo weapon), tentar GradientBoosting com mais árvores, e considerar Transformer/TFT para multi-horizon.